# 01 - Data Inception

**Goal:** get to know both Kaggle datasets, find problems in them, clean them, and save clean copies for the next notebooks.

| Dataset | Target (what we predict) | Rows |
|---|---|---|
| Fish Market | `Weight` of a fish (grams) | 159 |
| Body Fat | `BodyFat` percentage of a man | 252 |

Steps: load -> look at structure -> check quality -> clean -> save.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

fish_raw = pd.read_csv(RAW / "fish_market.csv")
bf_raw = pd.read_csv(RAW / "bodyfat.csv")
print("Fish market :", fish_raw.shape)
print("Body fat    :", bf_raw.shape)

Fish market : (159, 7)
Body fat    : (252, 15)


## 1. Fish Market - first look

In [2]:
fish_raw.head()

,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


In [3]:
fish_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Species  159 non-null    str    
 1   Weight   159 non-null    float64
 2   Length1  159 non-null    float64
 3   Length2  159 non-null    float64
 4   Length3  159 non-null    float64
 5   Height   159 non-null    float64
 6   Width    159 non-null    float64
dtypes: float64(6), str(1)
memory usage: 8.8 KB


In [4]:
fish_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
Weight,159.0,398.326415,357.978317,0.0000,120.00000,273.0000,650.0000,1650.000
Length1,159.0,26.247170,9.996441,7.5000,19.05000,25.2000,32.7000,59.000
Length2,159.0,28.415723,10.716328,8.4000,21.00000,27.3000,35.5000,63.400
Length3,159.0,31.227044,11.610246,8.8000,23.15000,29.4000,39.6500,68.000
Height,159.0,8.970994,4.286208,1.7284,5.94480,7.7860,12.3659,18.957
Width,159.0,4.417486,1.685804,1.0476,3.38565,4.2485,5.5845,8.142


**Column meaning (Fish Market)**

| Column | Meaning | Unit |
|---|---|---|
| `Species` | type of fish (category) | - |
| `Weight` | weight of the fish - **target** | grams |
| `Length1` | vertical length | cm |
| `Length2` | diagonal length | cm |
| `Length3` | cross length | cm |
| `Height` | height of the fish | cm |
| `Width` | diagonal width | cm |

In [5]:
print("Missing values per column:")
print(fish_raw.isna().sum())
print("\nDuplicate rows:", fish_raw.duplicated().sum())
print("\nFish per species:")
print(fish_raw["Species"].value_counts())

Missing values per column:
Species    0
Weight     0
Length1    0
Length2    0
Length3    0
Height     0
Width      0
dtype: int64

Duplicate rows: 0

Fish per species:
Species
Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6
Name: count, dtype: int64


The `min` of `Weight` in the table above is **0**. A fish cannot weigh 0 grams, so let's find it.

In [6]:
fish_raw[fish_raw["Weight"] <= 0]

,Species,Weight,Length1,Length2,Length3,Height,Width
40,Roach,0.0,19.0,20.5,22.8,6.4752,3.3516


One Roach has weight 0 but normal lengths. This is a data entry error, so we will remove that row.

## 2. Body Fat - first look

In [7]:
bf_raw.head()

,Density,BodyFat,Age,Weight,Height,Neck,Chest,Abdomen,Hip,Thigh,Knee,Ankle,Biceps,Forearm,Wrist
0,1.0708,12.3,23,154.25,67.75,36.2,93.1,85.2,94.5,59.0,37.3,21.9,32.0,27.4,17.1
1,1.0853,6.1,22,173.25,72.25,38.5,93.6,83.0,98.7,58.7,37.3,23.4,30.5,28.9,18.2
2,1.0414,25.3,22,154.00,66.25,34.0,95.8,87.9,99.2,59.6,38.9,24.0,28.8,25.2,16.6
3,1.0751,10.4,26,184.75,72.25,37.4,101.8,86.4,101.2,60.1,37.3,22.8,32.4,29.4,18.2
4,1.0340,28.7,24,184.25,71.25,34.4,97.3,100.0,101.9,63.2,42.2,24.0,32.2,27.7,17.7


In [8]:
bf_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Density  252 non-null    float64
 1   BodyFat  252 non-null    float64
 2   Age      252 non-null    int64  
 3   Weight   252 non-null    float64
 4   Height   252 non-null    float64
 5   Neck     252 non-null    float64
 6   Chest    252 non-null    float64
 7   Abdomen  252 non-null    float64
 8   Hip      252 non-null    float64
 9   Thigh    252 non-null    float64
 10  Knee     252 non-null    float64
 11  Ankle    252 non-null    float64
 12  Biceps   252 non-null    float64
 13  Forearm  252 non-null    float64
 14  Wrist    252 non-null    float64
dtypes: float64(14), int64(1)
memory usage: 29.7 KB


In [9]:
bf_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
Density,252.0,1.055574,0.019031,0.995,1.0414,1.0549,1.0704,1.1089
BodyFat,252.0,19.150794,8.368740,0.000,12.4750,19.2000,25.3000,47.5000
Age,252.0,44.884921,12.602040,22.000,35.7500,43.0000,54.0000,81.0000
Weight,252.0,178.924405,29.389160,118.500,159.0000,176.5000,197.0000,363.1500
Height,252.0,70.148810,3.662856,29.500,68.2500,70.0000,72.2500,77.7500
Neck,252.0,37.992063,2.430913,31.100,36.4000,38.0000,39.4250,51.2000
Chest,252.0,100.824206,8.430476,79.300,94.3500,99.6500,105.3750,136.2000
Abdomen,252.0,92.555952,10.783077,69.400,84.5750,90.9500,99.3250,148.1000
Hip,252.0,99.904762,7.164058,85.000,95.5000,99.3000,103.5250,147.7000
Thigh,252.0,59.405952,5.249952,47.200,56.0000,59.0000,62.3500,87.3000


**Column meaning (Body Fat)**

| Column | Meaning | Unit |
|---|---|---|
| `Density` | body density from underwater weighing | g/cm3 |
| `BodyFat` | body fat percentage - **target** | % |
| `Age` | age | years |
| `Weight` | body weight | pounds |
| `Height` | height | inches |
| `Neck`, `Chest`, `Abdomen`, `Hip`, `Thigh`, `Knee`, `Ankle`, `Biceps`, `Forearm`, `Wrist` | circumference measurements | cm |

In [10]:
print("Missing values:", int(bf_raw.isna().sum().sum()))
print("Duplicate rows:", bf_raw.duplicated().sum())

Missing values: 0
Duplicate rows: 0


### Problem 1: `Density` leaks the answer

Body fat % is *calculated* from density with the Siri equation:

`BodyFat = 495 / Density - 450`

If we keep `Density` as an input, the model would just rediscover this formula and look perfect, but it would be useless in real life because nobody knows a person's density without a lab test. So **`Density` must not be used as a feature.**

We can still use it for one thing: checking whether the `BodyFat` values are consistent.

In [11]:
bf = bf_raw.copy()
bf["siri_bf"] = 495 / bf["Density"] - 450
bf["gap"] = (bf["siri_bf"] - bf["BodyFat"]).abs()

print("Correlation between Density and BodyFat:", round(bf["Density"].corr(bf["BodyFat"]), 3))
print("\nRows where recorded BodyFat disagrees most with the Siri equation:")
bf.sort_values("gap", ascending=False)[["Density", "BodyFat", "siri_bf", "gap", "Height", "Weight"]].head(8)

Correlation between Density and BodyFat: -0.988

Rows where recorded BodyFat disagrees most with the Siri equation:


,Density,BodyFat,siri_bf,gap,Height,Weight
95,1.0991,17.4,0.368483,17.031517,77.75,224.50
47,1.0665,5.6,14.135021,8.535021,71.25,148.50
75,1.0666,18.5,14.091506,4.408494,67.50,148.25
181,1.1089,0.0,-3.611687,3.611687,68.00,118.50
168,1.0180,34.3,36.247544,1.947544,69.50,228.25
199,1.0462,23.6,23.140891,0.459109,67.50,170.75
5,1.0502,20.9,21.338793,0.438793,74.75,210.25
138,1.0481,22.4,22.283179,0.116821,71.25,168.25


For almost every row the gap is below 2 points, but a few rows are clearly wrong (for example a person with 17.4% body fat but a density that implies about 0.4%).
We treat rows with a gap above 3 points as recording errors.

### Problem 2: impossible height

In [12]:
bf_raw[bf_raw["Height"] < 50][["Age", "Weight", "Height", "Chest", "Abdomen"]]

,Age,Weight,Height,Chest,Abdomen
41,44,205.0,29.5,106.0,104.3


A man weighing 205 lb with a height of 29.5 inches is impossible (this is a typing error, probably 69.5). We remove it.

## 3. Cleaning

In [13]:
# ---- Fish: drop the zero-weight fish ----
fish = fish_raw[fish_raw["Weight"] > 0].reset_index(drop=True)

# ---- Body fat: drop inconsistent rows + impossible height, then drop Density (leakage) ----
bad_rows = (bf["gap"] > 3) | (bf["Height"] < 50)
print("Body fat rows removed:", int(bad_rows.sum()))
bf_clean = (
    bf.loc[~bad_rows]
      .drop(columns=["Density", "siri_bf", "gap"])
      .reset_index(drop=True)
)

summary = pd.DataFrame({
    "rows before": [len(fish_raw), len(bf_raw)],
    "rows after":  [len(fish), len(bf_clean)],
    "columns after": [fish.shape[1], bf_clean.shape[1]],
}, index=["Fish Market", "Body Fat"])
summary["rows removed"] = summary["rows before"] - summary["rows after"]
summary

Body fat rows removed:

 5


,rows before,rows after,columns after,rows removed
Fish Market,159,158,7,1
Body Fat,252,247,14,5


In [14]:
print("Fish min weight after cleaning    :", fish["Weight"].min())
print("Body fat min height after cleaning:", bf_clean["Height"].min())
print("Body fat columns:", list(bf_clean.columns))

Fish min weight after cleaning    : 5.9
Body fat min height after cleaning: 64.0
Body fat columns: ['BodyFat', 'Age', 'Weight', 'Height', 'Neck', 'Chest', 'Abdomen', 'Hip', 'Thigh', 'Knee', 'Ankle', 'Biceps', 'Forearm', 'Wrist']


## 4. Save clean data

In [15]:
fish.to_csv(OUT / "fish_clean.csv", index=False)
bf_clean.to_csv(OUT / "bodyfat_clean.csv", index=False)
print("Saved:", sorted(p.name for p in OUT.iterdir()))

Saved: ['bodyfat_clean.csv', 'fish_clean.csv']


## Summary

| | Fish Market | Body Fat |
|---|---|---|
| Missing values | none | none |
| Duplicates | none | none |
| Problems found | 1 fish with weight 0 | `Density` leaks the target; 4 rows inconsistent with it; 1 impossible height |
| Rows kept | 158 | 247 |
| Next step | `02_eda.ipynb` | `02_eda.ipynb` |